# VLM Benchmark Framework
Run the LLaVA-1.5 baseline evaluation for POPE and CHAIR directly in Google Colab.

**Note:** Make sure to go to `Runtime > Change runtime type` and select a **T4 GPU** (or better) before running.

In [ ]:
# 1. Clone your repository
!git clone https://github.com/dzungnguyen21/vlm_testingfield.git
%cd vlm_testingfield

In [ ]:
# 2. Install required dependencies
!pip install -q transformers accelerate datasets nltk Pillow

In [ ]:
# 3. Optionally login to Hugging Face if you encounter rate limits or gated datasets
from huggingface_hub import login
import os
# login('hf_....') # Replace with your token and uncomment to login

## Evaluate POPE

In [ ]:
# Generate outputs for POPE (limiting to 50 for quick test)
!python solution/baseline.py --run_pope --limit 50

In [ ]:
# Calculate POPE metrics based on generation
!python eval/pope.py --input_file results/baseline_pope.json --output_file results/eval_pope_baseline.json

## Evaluate CHAIR (Requires COCO Val2014)

In [ ]:
!pip install pycocotools -q
import os, requests, json
from tqdm.auto import tqdm
from pycocotools.coco import COCO

os.makedirs("coco/annotations", exist_ok=True)
os.makedirs("coco/images/val2014", exist_ok=True)

# Download COCO annotations
!wget -q -O coco/annotations_trainval2014.zip http://images.cocodataset.org/annotations/annotations_trainval2014.zip
!unzip -q -o coco/annotations_trainval2014.zip -d coco/

coco = COCO("coco/annotations/instances_val2014.json")
img_ids = coco.getImgIds()[:50]
imgs = coco.loadImgs(img_ids)

print("Downloading COCO val2014 images...")
for img_info in tqdm(imgs):
    save_path = f"coco/images/val2014/{img_info['file_name']}"
    if not os.path.exists(save_path):
        url = img_info["coco_url"]
        response = requests.get(url, timeout=10)
        with open(save_path, "wb") as f:
            f.write(response.content)
print(f"Downloaded {len(imgs)} images.")


In [ ]:
!python solution/baseline.py --run_chair --limit 50 --coco_dir coco/images/val2014 --coco_annotations coco/annotations/instances_val2014.json

In [ ]:
# Evaluate CHAIR based on generated captions
!python eval/chair.py --input_file results/baseline_chair.json --output_file results/eval_chair_baseline.json --coco_annotations coco/annotations/instances_val2014.json

## Evaluate H3 Attention Amplified on POPE

In [ ]:
# Generate outputs for POPE using H3 Attention Amplified (limiting to 50 for quick test)
!python solution/benchmark_h3_attention_amplified.py --run_pope --limit 50 --coco_annotations coco/annotations/instances_val2014.json --coco_dir coco/images/val2014

In [ ]:
# Calculate POPE metrics based on H3 generation
!python eval/pope.py --input_file results/h3_attention_amplified_pope.json --output_file results/eval_pope_h3.json

## Evaluate H3 Attention Amplified on CHAIR

In [ ]:
# Generate outputs for CHAIR using H3 Attention Amplified (limiting to 50 for quick test)
!python solution/benchmark_h3_attention_amplified.py --run_chair --limit 50 --coco_dir coco/images/val2014 --coco_annotations coco/annotations/instances_val2014.json

In [ ]:
# Evaluate CHAIR based on H3 generated captions
!python eval/chair.py --input_file results/h3_attention_amplified_chair.json --output_file results/eval_chair_h3.json --coco_annotations coco/annotations/instances_val2014.json